# Phase 1 — Kaggle T4, TTT-Linear 30M from scratch

Matched TinyStories run vs the **old attention snapshot (val PPL 5.23)**, not epoch-19 `last.pt` (PPL 7.00).

**Dataset:** New Dataset titled **`ttt-phase1-src`** (never `prototype`). Upload only `ttt-phase1-src.zip`. Do not upload the `prototype` folder.

`FROM_SCRATCH = True`. Do **not** resume `baseline-30m/last.pt`.

Settings: **GPU T4**, **Internet ON**. New notebook, not the attention continue-fit session.

Download `/kaggle/working/checkpoints/ttt-linear-30m/` after Save Version.

In [ ]:
!nvidia-smi -L
import torch
from pathlib import Path

assert torch.cuda.is_available(), "Settings → Accelerator → GPU T4, then Restart."
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import shutil, zipfile, os, sys

INP = Path("/kaggle/input")
assert INP.exists() and any(INP.iterdir()), "Add Input: dataset ttt-phase1-src (the zip), not a folder named prototype"

def first_zip():
    names = ("ttt-phase1-src.zip", "colab_upload.zip", "src30m.zip", "ttt-code.zip", "prototype.zip")
    for name in names:
        hits = list(INP.rglob(name))
        if hits:
            return hits[0]
    return None

train_py = [p for p in INP.rglob("scripts/train.py") if "__MACOSX" not in str(p)]
if train_py:
    os.chdir(train_py[0].parent.parent)
    print("using extracted dataset", Path.cwd())
else:
    code_zip = first_zip()
    assert code_zip is not None, "Need ttt-phase1-src.zip on the attached dataset"
    print("code zip", code_zip)
    extract_code = Path("/kaggle/working/from_code_zip")
    if extract_code.exists():
        shutil.rmtree(extract_code)
    extract_code.mkdir(parents=True)
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall(extract_code)
    matches = [p for p in extract_code.rglob("scripts/train.py") if "__MACOSX" not in str(p)]
    assert matches, "zip has no scripts/train.py"
    os.chdir(matches[0].parent.parent)
    print("working dir", Path.cwd())

sys.path.insert(0, str(Path.cwd()))
out = Path("/kaggle/working/checkpoints/ttt-linear-30m")
out.mkdir(parents=True, exist_ok=True)
train_npy = Path("data/tinystories-v2/train.npy")
assert train_npy.exists(), f"missing {train_npy} under {Path.cwd()}"
print("READY", "train.npy MB", round(train_npy.stat().st_size / 1e6, 1))
print("Phase 1 out", out)

In [ ]:
%pip install -q tokenizers numpy
from pathlib import Path

# True = Phase 1 from scratch (locked). Set False only to continue THIS TTT run after a Kaggle timeout.
FROM_SCRATCH = True
out = Path("/kaggle/working/checkpoints/ttt-linear-30m")
resume = out / "last.pt"
assert Path("scripts/train.py").exists()
print("FROM_SCRATCH", FROM_SCRATCH, "existing last.pt", resume.exists())

if FROM_SCRATCH:
    !python -u scripts/train.py --mixer ttt_linear --epochs 10 --optimizer adamw --weight-decay 0.1 \
      --dropout 0.1 --from-scratch --device cuda --amp auto --seq-len 256 --batch-size 2 --grad-accum 2 \
      --data data/tinystories-v2/train.npy --val-data data/tinystories-v2/validation.npy \
      --out {out}
else:
    resume_flag = f"--resume {resume}" if resume.exists() else ""
    !python -u scripts/train.py --mixer ttt_linear --epochs 10 --optimizer adamw --weight-decay 0.1 \
      --dropout 0.1 --device cuda --amp auto --seq-len 256 --batch-size 2 --grad-accum 2 \
      --data data/tinystories-v2/train.npy --val-data data/tinystories-v2/validation.npy \
      --out {out} {resume_flag}